# 🚀 Full Ethereum Data Export (No Sampling)

**Downloads COMPLETE 7-day Ethereum transaction data from BigQuery**

Expected data:
- ~1.2 million transactions per day
- ~8+ million transactions for 7 days
- ~2-4 GB total file size

Data is exported day-by-day to Parquet files and saved to Google Drive.

### Instructions:
1. Open in Colab: Runtime → Change runtime type → T4 GPU (optional, speeds up)
2. Run all cells
3. Download the parquet files from your Google Drive

In [ ]:
# Install required packages
!pip install -q google-cloud-bigquery db-dtypes pandas pyarrow tqdm
print("✅ Dependencies installed")

In [ ]:
# Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()
print("✅ Authenticated with Google Cloud")

In [ ]:
# Mount Google Drive for saving files
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = "/content/drive/MyDrive/eth_full_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Output directory: {OUTPUT_DIR}")

In [ ]:
# Configuration - EDIT THESE VALUES
DAYS_TO_FETCH = 7           # Number of days (1-30)
MIN_VALUE_ETH = 0.01        # Filter transactions below this value (0 = all)
INCLUDE_ZERO_VALUE = False  # Include 0-value transactions (contract calls)

print(f"📅 Will fetch {DAYS_TO_FETCH} days of data")
print(f"💰 Minimum value: {MIN_VALUE_ETH} ETH")

In [ ]:
# Estimate data size first
from google.cloud import bigquery
from datetime import datetime, timedelta

client = bigquery.Client()

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=DAYS_TO_FETCH)

value_filter = f"AND CAST(value AS FLOAT64) / 1e18 >= {MIN_VALUE_ETH}" if MIN_VALUE_ETH > 0 else ""

estimate_query = f"""
SELECT 
    COUNT(*) as total_transactions,
    ROUND(SUM(CAST(value AS FLOAT64)) / 1e18, 2) as total_eth,
    COUNT(DISTINCT from_address) as unique_senders,
    COUNT(DISTINCT to_address) as unique_receivers
FROM `bigquery-public-data.crypto_ethereum.transactions`
WHERE 
    block_timestamp >= TIMESTAMP('{start_date.strftime('%Y-%m-%d')}')
    AND block_timestamp < TIMESTAMP('{end_date.strftime('%Y-%m-%d')}')
    AND to_address IS NOT NULL
    {value_filter}
"""

print("📊 Estimating data size...")
result = client.query(estimate_query).result()
row = list(result)[0]

print(f"\n{'='*50}")
print(f"DATA ESTIMATE ({start_date.date()} to {end_date.date()})")
print(f"{'='*50}")
print(f"Total transactions: {row.total_transactions:,}")
print(f"Total ETH moved: {row.total_eth:,.2f}")
print(f"Unique senders: {row.unique_senders:,}")
print(f"Unique receivers: {row.unique_receivers:,}")
print(f"Estimated file size: ~{row.total_transactions * 500 / (1024**3):.2f} GB")
print(f"{'='*50}\n")

In [ ]:
# Export function - one day at a time
import pandas as pd
from tqdm.notebook import tqdm
import gc

def export_day(date, output_dir, min_value_eth=0.01):
    """Export all transactions for a single day."""
    date_str = date.strftime('%Y-%m-%d')
    next_date = date + timedelta(days=1)
    next_date_str = next_date.strftime('%Y-%m-%d')
    
    output_file = f"{output_dir}/eth_transactions_{date_str}.parquet"
    
    # Skip if already exists
    if os.path.exists(output_file):
        existing_df = pd.read_parquet(output_file)
        print(f"  {date_str}: Already exists ({len(existing_df):,} rows)")
        return len(existing_df)
    
    value_filter = f"AND CAST(value AS FLOAT64) / 1e18 >= {min_value_eth}" if min_value_eth > 0 else ""
    
    query = f"""
    SELECT
        `hash` as tx_hash,
        block_number,
        block_timestamp,
        from_address,
        to_address,
        CAST(value AS FLOAT64) / 1e18 as value_eth,
        CAST(gas_price AS FLOAT64) / 1e9 as gas_price_gwei,
        receipt_gas_used as gas_used,
        gas as gas_limit,
        COALESCE(transaction_type, 0) as transaction_type,
        nonce,
        transaction_index
    FROM `bigquery-public-data.crypto_ethereum.transactions`
    WHERE 
        block_timestamp >= TIMESTAMP('{date_str}')
        AND block_timestamp < TIMESTAMP('{next_date_str}')
        AND to_address IS NOT NULL
        {value_filter}
    ORDER BY block_number, transaction_index
    """
    
    print(f"  {date_str}: Fetching...")
    
    # Fetch data
    df = client.query(query).to_dataframe()
    
    if len(df) == 0:
        print(f"  {date_str}: No data found")
        return 0
    
    # Save to Parquet with compression
    df.to_parquet(output_file, compression='snappy', index=False)
    
    file_size_mb = os.path.getsize(output_file) / (1024**2)
    print(f"  {date_str}: ✅ Saved {len(df):,} transactions ({file_size_mb:.1f} MB)")
    
    # Free memory
    del df
    gc.collect()
    
    return len(df)

print("✅ Export function ready")

In [ ]:
# EXPORT ALL DATA - This will take 10-30 minutes depending on data size
print(f"\n{'='*60}")
print("🚀 STARTING FULL DATA EXPORT")
print(f"{'='*60}\n")

total_rows = 0
days_processed = 0
current_date = start_date

while current_date < end_date:
    rows = export_day(current_date, OUTPUT_DIR, MIN_VALUE_ETH)
    total_rows += rows
    days_processed += 1
    current_date += timedelta(days=1)

print(f"\n{'='*60}")
print("✅ EXPORT COMPLETE")
print(f"{'='*60}")
print(f"Total transactions: {total_rows:,}")
print(f"Days processed: {days_processed}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# Create manifest file
import json

manifest = {
    "export_date": datetime.now().isoformat(),
    "start_date": start_date.strftime('%Y-%m-%d'),
    "end_date": end_date.strftime('%Y-%m-%d'),
    "days_processed": days_processed,
    "total_rows": total_rows,
    "min_value_eth": MIN_VALUE_ETH,
    "files": [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.parquet')]
}

with open(f"{OUTPUT_DIR}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\n📄 Manifest created:")
print(json.dumps(manifest, indent=2))

In [ ]:
# List all exported files
print("\n📁 Exported files:")
print(f"{'='*60}")

total_size = 0
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size_mb = os.path.getsize(fpath) / (1024**2)
    total_size += size_mb
    print(f"  {f}: {size_mb:.1f} MB")

print(f"{'='*60}")
print(f"Total size: {total_size:.1f} MB ({total_size/1024:.2f} GB)")
print(f"\n✅ Files saved to Google Drive: {OUTPUT_DIR}")
print("\n💡 Download the parquet files to your local machine and run:")
print("   python scripts/batch_fraud_pipeline.py --input ./data/eth_full --output ./processed")

In [ ]:
# Quick preview of one file
files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.parquet')]
if files:
    sample_file = os.path.join(OUTPUT_DIR, files[0])
    df_sample = pd.read_parquet(sample_file)
    print(f"\n📋 Sample from {files[0]}:")
    print(f"Columns: {list(df_sample.columns)}")
    print(f"\nFirst 5 rows:")
    display(df_sample.head())
    print(f"\nValue distribution:")
    print(df_sample['value_eth'].describe())

---
## 📥 Download Instructions

1. Go to [Google Drive](https://drive.google.com)
2. Navigate to `My Drive > eth_full_data`
3. Download all `.parquet` files
4. Place them in `c:\amttp\data\eth_full\`
5. Run the batch pipeline:

```powershell
python scripts/batch_fraud_pipeline.py --input c:\amttp\data\eth_full --output c:\amttp\processed
```

Or use the **Colab to Local sync** below to download directly.

In [ ]:
# OPTIONAL: Zip all files for easier download
import shutil

zip_path = f"{OUTPUT_DIR}/eth_full_data.zip"
print("📦 Creating zip file...")

# Create zip of parquet files only
with shutil.ZipFile(zip_path, 'w', shutil.ZIP_DEFLATED) as zipf:
    for f in os.listdir(OUTPUT_DIR):
        if f.endswith('.parquet'):
            zipf.write(os.path.join(OUTPUT_DIR, f), f)

zip_size = os.path.getsize(zip_path) / (1024**3)
print(f"✅ Zip created: {zip_path}")
print(f"   Size: {zip_size:.2f} GB")